# Feature Selection

We select from a ranked list of features resulting from the algorithm developed in [DMEPOS_Feature_Ranking.ipynb](./DMEPOS_Feature_Ranking.ipynb) using ANOVA on 5-fold-by-provider cross validation area under the precision-recall curve (AUPRC) scores to determine the minimum number of useful features at least as good as a model built with all features as suggested by Hancock, J. T., Bauder, R. A., Wang, H., & Khoshgoftaar, T. M. (2023).

## WARNING! The following file must be run before using this script.
* SpIn_2_Artifacts/dmepos_amount_stats_labeled_silver.ipynb

In [1]:
import sys
print(sys.executable)

/cluster/software/conda-envs-global/jupyter/envs/jupytern_706/bin/python


In [2]:
%pip install scikit-learn

Defaulting to user installation because normal site-packages is not writeable
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 59.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 309.1/309.1 kB 43.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 63.6 MB/s eta 0:00:00a 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [3]:
%pip install catboost

Defaulting to user installation because normal site-packages is not writeable
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.2/114.2 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.2/97.2 MB 43.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.3/47.3 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 54.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 54.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 355.2/355.2 kB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 51.5 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 54.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 444.9/444.9 kB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 53.3 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━

In [2]:
%pip install xgboost

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.7/131.7 MB 39.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 289.8/289.8 MB 30.4 MB/s eta 0:00:0000:0100:01
Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install lightgbm

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 24.9 MB/s eta 0:00:0000:0100:01
Note: you may need to restart the kernel to use updated packages.


## Load modules

In [1]:
import pandas as pd
import numpy as np
import re
from catboost import CatBoostClassifier, Pool
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
import lightgbm as lgb
from sklearn.tree import DecisionTreeClassifier

## Load and transform data

In [4]:
df = pd.read_csv('/cluster/pixstor/haithcoatt-lab/SP26cCapStone_Team2/Data/Silver/dmepos_amount_stats_labeled.csv') # load

df.head()

,npi,rfrg_prvdr_state_abrvtn,year,target,avg_suplr_sbmtd_chrg_mean,avg_suplr_mdcr_alowd_amt_mean,avg_suplr_mdcr_pymt_amt_mean,avg_suplr_mdcr_stdzd_amt_mean,avg_suplr_sbmtd_chrg_sum,avg_suplr_mdcr_alowd_amt_sum,...,rfrg_prvdr_spclty_desc_student_in_an_organized_health_care_education_training_program,rfrg_prvdr_spclty_desc_surgery,rfrg_prvdr_spclty_desc_surgical_oncology,rfrg_prvdr_spclty_desc_thoracic_surgery,rfrg_prvdr_spclty_desc_thoracic_surgery_cardiothoracic_vascular_surgery,rfrg_prvdr_spclty_desc_undefined_physician_type,rfrg_prvdr_spclty_desc_undersea_and_hyperbaric_medicine,rfrg_prvdr_spclty_desc_unknown_supplier_provider_specialty,rfrg_prvdr_spclty_desc_urology,rfrg_prvdr_spclty_desc_vascular_surgery
0,1003000597,OK,2022,0,6.155416,2.196826,1.692306,1.702371,6.155416,2.196826,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1,1003000597,OK,2023,0,12.431501,6.574868,4.766645,5.263697,62.157505,32.874340,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
2,1003000902,KY,2021,0,97.751851,26.709274,19.463974,18.978969,782.014805,213.674191,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,1003000902,KY,2022,0,80.839496,18.260912,13.472218,12.561403,404.197480,91.304560,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,1003000902,KY,2023,0,39.630693,14.071564,10.200556,10.291238,158.522774,56.286255,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [5]:
X_train = df.drop(columns=['npi','rfrg_prvdr_state_abrvtn','year','target']).fillna(0) # the NaNs derive from insufficient data to compute std devs for some provider features
groups = df.npi # store npi in groups to ensure the same individual cannot be in both training and validation during cv
y_train = df.target
feature_names = X_train.columns.tolist()

## Feature Ranking algorithm

In [6]:
def get_rankings(X_train, y_train, classifier_array):
    """Returns a dataframe of the rank of each feature (row) in the ranking supplied
    by the given classifier algorithm(s) (column)."""
    df_rankings = pd.DataFrame(index=X_train.columns)
    # NOTE: if using XGB then StandardScaler must be applied
    for model, name in zip(classifier_array, labels):
        # addn'l preprocessing for XGB only
        if name == 'XGBClassifier':
            # apply standard scaler
            scaler = StandardScaler()
            X_train_scaled = scaler.fit_transform(X_train)
            # and train on the scaled data
            model.fit(X_train_scaled, y_train)
        else:
            # perform the usual training
            model.fit(X_train, y_train)

        # form a series of feature importance values indexed by feature_names
        feature_import_series = pd.Series(model.feature_importances_, index=feature_names).sort_values(ascending=False)
        # convert into a ranking and insert into the dataframe
        df_rankings[name] = feature_import_series.index.get_indexer(X_train.columns)

    # calculate the median ranks for each feature and sort ascending
    df_rankings['median_rank'] = df_rankings.apply(np.median, axis=1)
    df_rankings.sort_values(by='median_rank', inplace=True)
    return df_rankings

#### This takes a significant amount of time to run.

In [7]:
# init the base learners
catboost = CatBoostClassifier(iterations=100, verbose=0) # parameters given by Google Search AI Overview
xgb = XGBClassifier()
rf = RandomForestClassifier(random_state=8090)
et = ExtraTreesClassifier(n_estimators=100, random_state=8090)
lgbm = lgb.LGBMClassifier(random_state=8090)
dt = DecisionTreeClassifier(random_state=8090)

# gather them
classifier_array = [catboost, xgb, rf, et, lgbm, dt]
labels = [clf.__class__.__name__ for clf in classifier_array]

# run the ranking algorithm
df_rankings = get_rankings(X_train, y_train, classifier_array)
df_rankings.head()

[LightGBM] [Info] Number of positive: 44, number of negative: 140783
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.094873 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 20527
[LightGBM] [Info] Number of data points in the train set: 140827, number of used features: 155
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.000312 -> initscore=-8.070785
[LightGBM] [Info] Start training from score -8.070785


,CatBoostClassifier,XGBClassifier,RandomForestClassifier,ExtraTreesClassifier,LGBMClassifier,DecisionTreeClassifier,median_rank
avg_suplr_sbmtd_chrg_sum,37,48,0,1,2,1,1.5
avg_suplr_sbmtd_chrg_max,23,47,3,8,7,2,7.5
avg_suplr_mdcr_stdzd_amt_sum,11,18,9,3,14,0,10.0
avg_suplr_sbmtd_chrg_mean,15,60,1,4,6,14,10.0
avg_suplr_mdcr_pymt_amt_sum,8,12,12,9,36,56,12.0


In [8]:
df_rankings.to_pickle('feature_rankings.pkl')

## Feature Selection

In [4]:
# get the ranking df
df_rankings = pd.read_pickle('feature_rankings.pkl')
df_rankings.head()

,CatBoostClassifier,XGBClassifier,RandomForestClassifier,ExtraTreesClassifier,LGBMClassifier,DecisionTreeClassifier,median_rank
avg_suplr_sbmtd_chrg_median,1,66,8,5,22,2,6.5
bene_avg_age,12,61,16,4,0,1,8.0
avg_suplr_sbmtd_chrg_min,10,64,12,1,1,50,11.0
avg_suplr_mdcr_stdzd_amt_min,34,28,6,7,19,7,13.0
avg_suplr_sbmtd_chrg_mean,45,27,4,13,14,0,13.5


In [5]:
# unique median_rank
median_ranks = df_rankings.median_rank.unique()
median_ranks

array([  6.5,   8. ,  11. ,  13. ,  13.5,  16.5,  19.5,  20. ,  20.5,
        21. ,  22.5,  23.5,  24. ,  25. ,  25.5,  26. ,  26.5,  27.5,
        29. ,  29.5,  30. ,  30.5,  32. ,  33. ,  33.5,  34. ,  36. ,
        36.5,  37.5,  40. ,  41. ,  41.5,  42. ,  42.5,  43.5,  44.5,
        45.5,  46. ,  46.5,  49. ,  49.5,  50. ,  52.5,  56. ,  57. ,
        57.5,  60.5,  61. ,  61.5,  62. ,  62.5,  65.5,  67. ,  68. ,
        68.5,  70.5,  71.5,  75. ,  76. ,  80. ,  81.5,  82.5,  83.5,
        84. ,  85. ,  87.5,  88.5,  90. ,  92.5,  93.5,  94. ,  95.5,
        98.5,  99.5, 100. , 105.5, 106.5, 110. , 111. , 111.5, 112. ,
       113. , 114. , 114.5, 116.5, 117. , 118. , 119.5, 120.5, 121. ,
       121.5, 123. , 123.5, 124.5, 125. , 126. , 127. , 128.5, 129. ,
       129.5, 130. , 131. , 132. , 132.5, 133. , 134. , 134.5, 135. ,
       137. , 137.5, 138.5, 139. , 139.5, 141. , 141.5, 142. , 144. ,
       151. , 152. , 153. , 153.5, 154. , 158.5, 161. , 162. , 163.5,
       164.5, 165. ,

In [6]:
# number of unique median_rank
df_rankings.median_rank.nunique()

172

In [7]:
# double check
len(median_ranks)

172

## Prototyping the selection algorithm

In [8]:
# necessary modules
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import precision_recall_curve, auc, average_precision_score

### CatBoost

In [11]:
sgkf = StratifiedGroupKFold(n_splits=5)

auprc_scores = []

# get feature set from df_ranking and filter X_train
feats = df_rankings[df_rankings.median_rank <= 6.5].index
X = X_train.filter(items=feats, axis='columns')

# the model
model = CatBoostClassifier(iterations=100, eval_metric='PRAUC', verbose=0)

for train_idx, val_idx in sgkf.split(X, y_train, groups):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

    # train and fit the model
    model.fit(X_tr, y_tr, eval_set=(X_val, y_val))
    
    # Calculate AUPRC on validation set
    preds = model.predict_proba(X_val)[:, 1]
    precision, recall, _ = precision_recall_curve(y_val, preds)
    auprc = auc(recall, precision)
    auprc_scores.append(auprc)

# need to store results w/ a column for classifier, num_features, auprc
cv_res = {'classifier' : [model.__class__.__name__]*5,
          'num_features' : [len(feats)]*5,
          'auprc' : auprc_scores}

df_cv_res = pd.DataFrame(data=cv_res)
df_cv_res

,classifier,num_features,auprc
0,CatBoostClassifier,1,0.045996
1,CatBoostClassifier,1,0.000575
2,CatBoostClassifier,1,0.000540
3,CatBoostClassifier,1,0.000442
4,CatBoostClassifier,1,0.000558


### XGBoost

In [19]:
sgkf = StratifiedGroupKFold(n_splits=5)

auprc_scores = []

# get feature set from df_ranking and filter X_train
feats = df_rankings[df_rankings.median_rank <= 6.5].index

model = XGBClassifier(eval_metric='aucpr')
X = X_train.filter(items=feats, axis='columns')

for train_idx, val_idx in sgkf.split(X, y_train, groups):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

    # train and fit the model
    model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)]) # eval_set needs to be a list for XGBoost
    
    # Calculate AUPRC on validation set
    preds = model.predict_proba(X_val)[:, 1]
    
    auprc = average_precision_score(y_val, preds)
    auprc_scores.append(auprc)

# need to store results w/ a column for classifier, num_features, auprc
cv_res = {'classifier' : [ model.__class__.__name__]*5,
          'num_features' : [len(feats)]*5,
          'auprc' : auprc_scores}

df_cv_res = pd.DataFrame(data=cv_res)
df_cv_res

[0]	validation_0-aucpr:0.00046
[1]	validation_0-aucpr:0.00042
[2]	validation_0-aucpr:0.00037
[3]	validation_0-aucpr:0.00036
[4]	validation_0-aucpr:0.00033
[5]	validation_0-aucpr:0.00033
[6]	validation_0-aucpr:0.00033
[7]	validation_0-aucpr:0.00030
[8]	validation_0-aucpr:0.00029
[9]	validation_0-aucpr:0.00031
[10]	validation_0-aucpr:0.00031
[11]	validation_0-aucpr:0.00029
[12]	validation_0-aucpr:0.00029
[13]	validation_0-aucpr:0.00029
[14]	validation_0-aucpr:0.00029
[15]	validation_0-aucpr:0.00029
[16]	validation_0-aucpr:0.00028
[17]	validation_0-aucpr:0.00028
[18]	validation_0-aucpr:0.00028
[19]	validation_0-aucpr:0.00028
[20]	validation_0-aucpr:0.00029
[21]	validation_0-aucpr:0.00028
[22]	validation_0-aucpr:0.00028
[23]	validation_0-aucpr:0.00028
[24]	validation_0-aucpr:0.00028
[25]	validation_0-aucpr:0.00029
[26]	validation_0-aucpr:0.00028
[27]	validation_0-aucpr:0.00028
[28]	validation_0-aucpr:0.00028
[29]	validation_0-aucpr:0.00028
[30]	validation_0-aucpr:0.00028
[31]	validation_0-

,classifier,num_features,auprc
0,XGBClassifier,1,0.000327
1,XGBClassifier,1,0.000365
2,XGBClassifier,1,0.000519
3,XGBClassifier,1,0.000366
4,XGBClassifier,1,0.000412


### Random Forest

In [21]:
sgkf = StratifiedGroupKFold(n_splits=5)

auprc_scores = []

# get feature set from df_ranking and filter X_train
feats = df_rankings[df_rankings.median_rank <= 6.5].index
X = X_train.filter(items=feats, axis='columns')

# the model
model = RandomForestClassifier()

for train_idx, val_idx in sgkf.split(X, y_train, groups):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

    # train and fit the model
    model.fit(X_tr, y_tr)
    
    # Calculate AUPRC on validation set
    preds = model.predict_proba(X_val)[:, 1]
    precision, recall, _ = precision_recall_curve(y_val, preds)
    auprc = auc(recall, precision)
    auprc_scores.append(auprc)

# need to store results w/ a column for classifier, num_features, auprc
cv_res = {'classifier' : [model.__class__.__name__]*5,
          'num_features' : [len(feats)]*5,
          'auprc' : auprc_scores}

df_cv_res = pd.DataFrame(data=cv_res)
df_cv_res

,classifier,num_features,auprc
0,RandomForestClassifier,1,0.000195
1,RandomForestClassifier,1,0.000195
2,RandomForestClassifier,1,0.000195
3,RandomForestClassifier,1,0.000213
4,RandomForestClassifier,1,0.000195


### Extremely Randomized Trees

In [22]:
sgkf = StratifiedGroupKFold(n_splits=5)

auprc_scores = []

# get feature set from df_ranking and filter X_train
feats = df_rankings[df_rankings.median_rank <= 6.5].index
X = X_train.filter(items=feats, axis='columns')

# the model
model = ExtraTreesClassifier(n_estimators=100)

for train_idx, val_idx in sgkf.split(X, y_train, groups):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

    # train and fit the model
    model.fit(X_tr, y_tr)
    
    # Calculate AUPRC on validation set
    preds = model.predict_proba(X_val)[:, 1]
    precision, recall, _ = precision_recall_curve(y_val, preds)
    auprc = auc(recall, precision)
    auprc_scores.append(auprc)

# need to store results w/ a column for classifier, num_features, auprc
cv_res = {'classifier' : [model.__class__.__name__]*5,
          'num_features' : [len(feats)]*5,
          'auprc' : auprc_scores}

df_cv_res = pd.DataFrame(data=cv_res)
df_cv_res

,classifier,num_features,auprc
0,ExtraTreesClassifier,1,0.000195
1,ExtraTreesClassifier,1,0.000195
2,ExtraTreesClassifier,1,0.000195
3,ExtraTreesClassifier,1,0.000213
4,ExtraTreesClassifier,1,0.000195


### LightGBM

In [24]:
sgkf = StratifiedGroupKFold(n_splits=5)

auprc_scores = []

# get feature set from df_ranking and filter X_train
feats = df_rankings[df_rankings.median_rank <= 6.5].index
X = X_train.filter(items=feats, axis='columns')

# the model
model = lgb.LGBMClassifier()

for train_idx, val_idx in sgkf.split(X, y_train, groups):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

    # train and fit the model
    model.fit(X_tr, y_tr)
    
    # Calculate AUPRC on validation set
    preds = model.predict_proba(X_val)[:, 1]
    precision, recall, _ = precision_recall_curve(y_val, preds)
    auprc = auc(recall, precision)
    auprc_scores.append(auprc)

# need to store results w/ a column for classifier, num_features, auprc
cv_res = {'classifier' : [model.__class__.__name__]*5,
          'num_features' : [len(feats)]*5,
          'auprc' : auprc_scores}

df_cv_res = pd.DataFrame(data=cv_res)
df_cv_res

[LightGBM] [Info] Number of positive: 45, number of negative: 112616
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002777 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 255
[LightGBM] [Info] Number of data points in the train set: 112661, number of used features: 1
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.000399 -> initscore=-7.825077
[LightGBM] [Info] Start training from score -7.825077
[LightGBM] [Info] Number of positive: 45, number of negative: 112617
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000172 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 255
[LightGBM] [Info] Number of data points in the train set: 112662, number of used features: 1
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.000399 -> initscore=-7.825085
[LightGB

,classifier,num_features,auprc
0,LGBMClassifier,1,0.000298
1,LGBMClassifier,1,0.000364
2,LGBMClassifier,1,0.000409
3,LGBMClassifier,1,0.000314
4,LGBMClassifier,1,0.000369


### Decision Tree

In [25]:
sgkf = StratifiedGroupKFold(n_splits=5)

auprc_scores = []

# get feature set from df_ranking and filter X_train
feats = df_rankings[df_rankings.median_rank <= 6.5].index
X = X_train.filter(items=feats, axis='columns')

# the model
model = DecisionTreeClassifier()

for train_idx, val_idx in sgkf.split(X, y_train, groups):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

    # train and fit the model
    model.fit(X_tr, y_tr)
    
    # Calculate AUPRC on validation set
    preds = model.predict_proba(X_val)[:, 1]
    precision, recall, _ = precision_recall_curve(y_val, preds)
    auprc = auc(recall, precision)
    auprc_scores.append(auprc)

# need to store results w/ a column for classifier, num_features, auprc
cv_res = {'classifier' : [model.__class__.__name__]*5,
          'num_features' : [len(feats)]*5,
          'auprc' : auprc_scores}

df_cv_res = pd.DataFrame(data=cv_res)
df_cv_res

,classifier,num_features,auprc
0,DecisionTreeClassifier,1,0.000195
1,DecisionTreeClassifier,1,0.000195
2,DecisionTreeClassifier,1,0.000195
3,DecisionTreeClassifier,1,0.000213
4,DecisionTreeClassifier,1,0.000195


## Iterate through the models with a for loop

In [10]:
# init the base learners
# NOTE: we will need to recover the random states in the final algorithm
catboost = CatBoostClassifier(iterations=100, eval_metric='PRAUC', verbose=0)
xgb = XGBClassifier(eval_metric='aucpr')
rf = RandomForestClassifier()
et = ExtraTreesClassifier(n_estimators=100)
lgbm = lgb.LGBMClassifier()
dt = DecisionTreeClassifier()

# gather them
classifier_array = [catboost, xgb, rf, et, lgbm, dt]

sgkf = StratifiedGroupKFold(n_splits=5)

df_all_cv_res = pd.DataFrame(columns=['classifier','num_features','auprc']) # empty dataframe to put results in

# get feature set from df_ranking and filter X_train
# NOTE: iterate through all median rank values in final algorithm
feats = df_rankings[df_rankings.median_rank <= 6.5].index
X = X_train.filter(items=feats, axis='columns')

for model in classifier_array:
    auprc_scores = []
    
    for train_idx, val_idx in sgkf.split(X, y_train, groups):
        X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

        if model.__class__.__name__ == 'XGBClassifier':
            # train and fit the model
            model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)]) # eval_set needs to be a list for XGBoost
            # Calculate AUPRC on validation set
            preds = model.predict_proba(X_val)[:, 1]
            auprc = average_precision_score(y_val, preds)
            auprc_scores.append(auprc)
            
        else:
            if model.__class__.__name__ == 'CatBoostClassifier':
                # train and fit the model
                model.fit(X_tr, y_tr, eval_set=(X_val, y_val))
            else:
                # train and fit the model
                model.fit(X_tr, y_tr)
            # Calculate AUPRC on validation set
            preds = model.predict_proba(X_val)[:, 1]
            precision, recall, _ = precision_recall_curve(y_val, preds)
            auprc = auc(recall, precision)
            auprc_scores.append(auprc)
    
    # need to store results w/ a column for classifier, num_features, auprc
    cv_res = {'classifier' : [model.__class__.__name__]*5,
              'num_features' : [len(feats)]*5,
              'auprc' : auprc_scores}
    df_cv_res = pd.DataFrame(data=cv_res)
    df_all_cv_res = pd.concat([df_all_cv_res, df_cv_res]).reset_index(drop=True)


C:\Users\camka\AppData\Local\Temp\ipykernel_22072\916417795.py:55: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_all_cv_res = pd.concat([df_all_cv_res, df_cv_res])


[0]	validation_0-aucpr:0.00046
[1]	validation_0-aucpr:0.00042
[2]	validation_0-aucpr:0.00037
[3]	validation_0-aucpr:0.00036
[4]	validation_0-aucpr:0.00033
[5]	validation_0-aucpr:0.00033
[6]	validation_0-aucpr:0.00033
[7]	validation_0-aucpr:0.00030
[8]	validation_0-aucpr:0.00029
[9]	validation_0-aucpr:0.00031
[10]	validation_0-aucpr:0.00031
[11]	validation_0-aucpr:0.00029
[12]	validation_0-aucpr:0.00029
[13]	validation_0-aucpr:0.00029
[14]	validation_0-aucpr:0.00029
[15]	validation_0-aucpr:0.00029
[16]	validation_0-aucpr:0.00028
[17]	validation_0-aucpr:0.00028
[18]	validation_0-aucpr:0.00028
[19]	validation_0-aucpr:0.00028
[20]	validation_0-aucpr:0.00029
[21]	validation_0-aucpr:0.00028
[22]	validation_0-aucpr:0.00028
[23]	validation_0-aucpr:0.00028
[24]	validation_0-aucpr:0.00028
[25]	validation_0-aucpr:0.00029
[26]	validation_0-aucpr:0.00028
[27]	validation_0-aucpr:0.00028
[28]	validation_0-aucpr:0.00028
[29]	validation_0-aucpr:0.00028
[30]	validation_0-aucpr:0.00028
[31]	validation_0-

In [ ]:
df_all_cv_res

## Turn the 5-fold CV subroutine into a function

In [12]:
def sgkf_cv(model, X, y_train, groups, n_splits=5):
    """Does stratified group k-fold cross validation where k=n_splits (default 5)
    then returns an array of scores with length n_splits."""
    sgkf=StratifiedGroupKFold(n_splits=n_splits)
    auprc_scores = []
    for train_idx, val_idx in sgkf.split(X, y_train, groups):
        X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
        if model.__class__.__name__ == 'XGBClassifier':
            # train and fit the model
            model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)]) # eval_set needs to be a list for XGBoost
            # Calculate AUPRC on validation set
            preds = model.predict_proba(X_val)[:, 1]
            auprc = average_precision_score(y_val, preds)
            auprc_scores.append(auprc)
        else:
            if model.__class__.__name__ == 'CatBoostClassifier':
                # train and fit the model
                model.fit(X_tr, y_tr, eval_set=(X_val, y_val))
            else:
                # train and fit the model
                model.fit(X_tr, y_tr)
            # Calculate AUPRC on validation set
            preds = model.predict_proba(X_val)[:, 1]
            precision, recall, _ = precision_recall_curve(y_val, preds)
            auprc = auc(recall, precision)
            auprc_scores.append(auprc)
    return auprc_scores

In [14]:
# init the base learners
# NOTE: we will need to recover the random states in the final algorithm
catboost = CatBoostClassifier(iterations=100, eval_metric='PRAUC', verbose=0)
xgb = XGBClassifier(eval_metric='aucpr')
rf = RandomForestClassifier()
et = ExtraTreesClassifier(n_estimators=100)
lgbm = lgb.LGBMClassifier()
dt = DecisionTreeClassifier()

# gather them
classifier_array = [catboost, xgb, rf, et, lgbm, dt]

sgkf = StratifiedGroupKFold(n_splits=5)

df_all_cv_res = pd.DataFrame(columns=['classifier','num_features','auprc']) # empty dataframe to put results in

# get feature set from df_ranking and filter X_train
# NOTE: iterate through all median rank values in final algorithm
feats = df_rankings[df_rankings.median_rank <= 6.5].index
X = X_train.filter(items=feats, axis='columns')

for model in classifier_array:
    n_splits = 5
    auprc_scores = sgkf_cv(model, X, y_train, groups, n_splits=n_splits)
    # need to store results w/ a column for classifier, num_features, auprc
    cv_res = {'classifier' : [model.__class__.__name__]*n_splits,
              'num_features' : [len(feats)]*n_splits,
              'auprc' : auprc_scores}
    df_cv_res = pd.DataFrame(data=cv_res)
    df_all_cv_res = pd.concat([df_all_cv_res, df_cv_res]).reset_index(drop=True)

C:\Users\camka\AppData\Local\Temp\ipykernel_22072\3164419889.py:30: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_all_cv_res = pd.concat([df_all_cv_res, df_cv_res]).reset_index(drop=True)


[0]	validation_0-aucpr:0.00046
[1]	validation_0-aucpr:0.00042
[2]	validation_0-aucpr:0.00037
[3]	validation_0-aucpr:0.00036
[4]	validation_0-aucpr:0.00033
[5]	validation_0-aucpr:0.00033
[6]	validation_0-aucpr:0.00033
[7]	validation_0-aucpr:0.00030
[8]	validation_0-aucpr:0.00029
[9]	validation_0-aucpr:0.00031
[10]	validation_0-aucpr:0.00031
[11]	validation_0-aucpr:0.00029
[12]	validation_0-aucpr:0.00029
[13]	validation_0-aucpr:0.00029
[14]	validation_0-aucpr:0.00029
[15]	validation_0-aucpr:0.00029
[16]	validation_0-aucpr:0.00028
[17]	validation_0-aucpr:0.00028
[18]	validation_0-aucpr:0.00028
[19]	validation_0-aucpr:0.00028
[20]	validation_0-aucpr:0.00029
[21]	validation_0-aucpr:0.00028
[22]	validation_0-aucpr:0.00028
[23]	validation_0-aucpr:0.00028
[24]	validation_0-aucpr:0.00028
[25]	validation_0-aucpr:0.00029
[26]	validation_0-aucpr:0.00028
[27]	validation_0-aucpr:0.00028
[28]	validation_0-aucpr:0.00028
[29]	validation_0-aucpr:0.00028
[30]	validation_0-aucpr:0.00028
[31]	validation_0-

In [15]:
df_all_cv_res

,classifier,num_features,auprc
0,CatBoostClassifier,1,0.045996
1,CatBoostClassifier,1,0.000575
2,CatBoostClassifier,1,0.000540
3,CatBoostClassifier,1,0.000442
4,CatBoostClassifier,1,0.000558
5,XGBClassifier,1,0.000327
6,XGBClassifier,1,0.000365
7,XGBClassifier,1,0.000519
8,XGBClassifier,1,0.000366
9,XGBClassifier,1,0.000412


## References

  Hancock, J. T., Bauder, R. A., Wang, H., & Khoshgoftaar, T. M. (2023). Explainable machine learning models for Medicare fraud detection. Journal of Big Data, 10(1), 154. https://doi.org/10.1186/s40537-023-00821-5